# Exportar máscaras preditas (Zenodo)

Carrega os pesos de cada rodada em `saida/runs/<run_id>/pesos.weights.h5`, prediz **todos** os recortes do conjunto de teste
(sem sorteio) e salva as máscaras binárias em `saida/mascaras_preditas/<run_id>/`. Ao final, gera um `.zip` para depósito no Zenodo.

O Dice impresso por rodada serve apenas como verificação de que os pesos carregados são os corretos: uma função estável com Dice próximo de zero indica pesos sobrescritos.


In [1]:
import os, glob, csv, zipfile
import numpy as np
from PIL import Image
import tensorflow as tf
from keras.models import Model
from keras.layers import Conv2D, MaxPooling2D, Input, Conv2DTranspose, SpatialDropout2D, concatenate


In [2]:
dataset     = "crop_OralEpitheliumDB_aug"
pasta_teste = os.path.join("datasets", dataset, "test")
pasta_runs  = os.path.join("saida", "runs")
pasta_saida = os.path.join("saida", "mascaras_preditas")
limiar      = 0.5


In [3]:
# Mesma arquitetura do Unet_ProtoSeg.ipynb (necessária para load_weights)
def unet(sz=(256, 256, 3)):
    inputs = Input(sz)
    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    conv1 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D((2, 2))(conv1)
    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(pool1)
    conv2 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv2)
    drop2 = SpatialDropout2D(0.50)(conv2)
    pool2 = MaxPooling2D((2, 2))(drop2)
    conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(pool2)
    conv3 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv3)
    drop3 = SpatialDropout2D(0.50)(conv3)
    pool3 = MaxPooling2D((2, 2))(drop3)
    conv4 = Conv2D(256, (3, 3), activation='relu', padding='same')(pool3)
    conv4 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv4)
    drop4 = SpatialDropout2D(0.50)(conv4)
    pool4 = MaxPooling2D((2, 2))(drop4)
    conv5 = Conv2D(512, (3, 3), activation='relu', padding='same')(pool4)
    conv5 = Conv2D(512, (3, 3), activation='relu', padding='same')(conv5)
    drop5 = SpatialDropout2D(0.50)(conv5)
    up6 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(drop5)
    merge6 = concatenate([up6, drop4], axis=3)
    conv6 = Conv2D(256, (3, 3), activation='relu', padding='same')(merge6)
    conv6 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv6)
    up7 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(conv6)
    merge7 = concatenate([up7, drop3], axis=3)
    conv7 = Conv2D(128, (3, 3), activation='relu', padding='same')(merge7)
    conv7 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv7)
    up8 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv7)
    merge8 = concatenate([up8, drop2], axis=3)
    conv8 = Conv2D(64, (3, 3), activation='relu', padding='same')(merge8)
    conv8 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv8)
    up9 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv8)
    merge9 = concatenate([up9], axis=3)
    conv9 = Conv2D(32, (3, 3), activation='relu', padding='same')(merge9)
    conv9 = Conv2D(32, (3, 3), activation='relu', padding='same')(conv9)
    output = Conv2D(1, (1, 1), activation='sigmoid')(conv9)
    return Model(inputs, output)


In [4]:
# Conjunto de teste completo, em ordem fixa, lido em RGB como no treinamento
test_files = sorted(os.path.splitext(f)[0] for f in os.listdir(pasta_teste)
                    if os.path.isfile(os.path.join(pasta_teste, f)))

def carrega(nome):
    img = np.array(Image.open(os.path.join(pasta_teste, f"{nome}.png")).resize((256, 256)))
    if img.ndim == 2:
        img = np.stack((img,) * 3, axis=-1)
    img = img[:, :, 0:3] / 255.
    msk = np.array(Image.open(os.path.join(pasta_teste, "mascaras", f"{nome}.png")).resize((256, 256)))
    if msk.ndim == 3:
        msk = msk[:, :, 0]
    return img, (msk > 0).astype(np.uint8)

X, Y = map(np.array, zip(*[carrega(n) for n in test_files]))
print("Recortes de teste:", len(test_files), X.shape, Y.shape)


Recortes de teste: 184 (184, 256, 256, 3) (184, 256, 256)


In [5]:
model = unet()
resumo = []
for run_dir in sorted(glob.glob(os.path.join(pasta_runs, "*"))):
    pesos = os.path.join(run_dir, "pesos.weights.h5")
    run_id = os.path.basename(run_dir)
    if not os.path.exists(pesos):
        print("SEM PESOS:", run_id)
        continue
    model.load_weights(pesos)
    prob = model.predict(X, batch_size=16, verbose=0)[..., 0]
    pred = (prob > limiar).astype(np.uint8)

    destino = os.path.join(pasta_saida, run_id)
    os.makedirs(destino, exist_ok=True)
    for nome, m in zip(test_files, pred):
        Image.fromarray(m * 255).save(os.path.join(destino, f"{nome}.png"))

    inter = (pred * Y).sum(axis=(1, 2))
    soma = pred.sum(axis=(1, 2)) + Y.sum(axis=(1, 2))
    dice = float(np.mean((2 * inter + 1e-6) / (soma + 1e-6)))
    resumo.append({"run_id": run_id, "mascaras": len(pred), "dice_teste_completo": round(dice, 4)})
    print(f"{run_id}: {len(pred)} máscaras | Dice (verificação) = {dice:.4f}")

with open(os.path.join(pasta_saida, "resumo_exportacao.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["run_id", "mascaras", "dice_teste_completo"])
    w.writeheader(); w.writerows(resumo)


Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-bce__seed123: 184 máscaras | Dice (verificação) = 0.7247
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-bce__seed2024: 184 máscaras | Dice (verificação) = 0.7383
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-bce__seed42: 184 máscaras | Dice (verificação) = 0.7257
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-combo__seed123: 184 máscaras | Dice (verificação) = 0.7344
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-combo__seed2024: 184 máscaras | Dice (verificação) = 0.7384
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-combo__seed42: 184 máscaras | Dice (verificação) = 0.7349
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-combo_pesada_bce_reverso__seed123: 184 máscaras | Dice (verificação) = 0.7306
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-combo_pesada_bce_reverso__seed2024: 184 máscaras | Dice (verificação) = 0.7327
Unet__crop_OralEpitheliumDB_aug__proto-offline__loss-combo_pesada_bce_r

In [6]:
# Um único .zip para o depósito no Zenodo
zip_path = os.path.join("saida", "mascaras_preditas_teste.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for raiz, _, arquivos in os.walk(pasta_saida):
        for a in arquivos:
            caminho = os.path.join(raiz, a)
            z.write(caminho, os.path.relpath(caminho, pasta_saida))
print(zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")


saida\mascaras_preditas_teste.zip 19.3 MB
